# 05. Comparación de resultados y figuras del informe

**MAP - Charting Student Math Misunderstandings**

En este cuaderno se reúnen las métricas de todos los modelos y se producen las visualizaciones
estáticas que acompañan al informe. Las figuras se guardan en `reports/figures/`.

Requiere haber ejecutado antes el cuaderno 03. El cuaderno 04 es opcional: si sus resultados
están disponibles, el transformer se incorpora a la comparación.

## 0. Arranque

Este cuaderno funciona igual en **Google Colab** y en una instalación local.

En Colab la celda siguiente clona el repositorio e instala las dependencias. Como las métricas
las produce el cuaderno 03, hay que subir `artefactos_modelado.zip` cuando se solicite.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPOSITORIO = "https://github.com/Tunchxz/math-misconception-map.git"
CARPETA = "math-misconception-map"

if EN_COLAB:
    if not Path(CARPETA).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPOSITORIO, CARPETA], check=True)
    os.chdir(f"/content/{CARPETA}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
    )

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Entorno:", "Google Colab" if EN_COLAB else "local")
print("Raíz del proyecto:", RAIZ)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import DATA_TRANSFORMED, FIGURES, PALETA_CUALITATIVA
from src.data.make_dataset import asegurar_datos, cargar_transformado
from src.visualization.visualize import guardar_figura, set_estilo

set_estilo()
pd.set_option("display.width", 170)
FIGURES.mkdir(parents=True, exist_ok=True)

if EN_COLAB and not (DATA_TRANSFORMED / "metricas_clasicos.parquet").exists():
    from google.colab import files

    print("Suba artefactos_modelado.zip, generado por el cuaderno 03")
    subidos = files.upload()
    import zipfile

    with zipfile.ZipFile(next(iter(subidos))) as z:
        z.extractall("artefactos")
    import shutil

    shutil.copytree("artefactos/transformed", DATA_TRANSFORMED, dirs_exist_ok=True)
    shutil.copytree("artefactos/models", RAIZ / "models", dirs_exist_ok=True)

metricas = cargar_transformado("metricas_clasicos")
comparacion = cargar_transformado("comparacion_texto_contexto")
train = asegurar_datos(verbose=False)

print("Filas de métricas:", len(metricas))
print("Modelos:", metricas["modelo"].nunique(), "| Etapas:", metricas["etapa"].nunique())

## 1. Colores del informe

Se usa la paleta de Okabe-Ito, construida para ser distinguible bajo los tres tipos de
daltonismo. Los dos esquemas de validación reciben un color fijo en todas las figuras, de modo
que su significado no cambie de un gráfico a otro.

In [ ]:
COLOR_ALEATORIO = "#56B4E9"
COLOR_AGRUPADO = "#D55E00"
COLOR_TEXTO = "#009E73"
COLOR_CONTEXTO = "#53606B"

COLOR_ESQUEMA = {"aleatorio": COLOR_ALEATORIO, "agrupado": COLOR_AGRUPADO}
ETIQUETA_ESQUEMA = {
    "aleatorio": "Aleatorio\n(preguntas ya vistas)",
    "agrupado": "Agrupado por pregunta\n(preguntas nuevas)",
}

orden_modelos = (
    metricas[metricas["etapa"] == "1. Category"]
    .groupby("modelo")["map@3"].max().sort_values().index.tolist()
)
print("Modelos ordenados por MAP@3:", orden_modelos)

## 2. Figura 1: comparación de los modelos en las tres etapas

In [ ]:
etapas = sorted(metricas["etapa"].unique())
fig, axes = plt.subplots(1, len(etapas), figsize=(15, 5.2), sharey=True)

for ax, etapa in zip(np.atleast_1d(axes), etapas):
    sub = metricas[metricas["etapa"] == etapa]
    pos = np.arange(len(orden_modelos))
    ancho = 0.38
    for i, esquema in enumerate(["aleatorio", "agrupado"]):
        valores = [
            sub[(sub["modelo"] == m) & (sub["esquema"] == esquema)]["map@3"].mean()
            for m in orden_modelos
        ]
        ax.barh(pos + (i - 0.5) * ancho, valores, height=ancho,
                color=COLOR_ESQUEMA[esquema], label=ETIQUETA_ESQUEMA[esquema].replace("\n", " "))
    ax.set_yticks(pos)
    ax.set_yticklabels(orden_modelos)
    ax.set_xlabel("MAP@3")
    ax.set_title(etapa)
    ax.set_xlim(0, 1)

axes[0].legend(loc="lower right", fontsize=8.5)
fig.suptitle("Rendimiento por modelo, etapa y esquema de validación",
             fontsize=13, fontweight="bold")
fig.tight_layout()
print(guardar_figura(fig, "fig1_comparacion_modelos"))
plt.show()

## 3. Figura 2: cuánta memorización de la pregunta hay

La diferencia entre validar con preguntas ya vistas y con preguntas nuevas mide directamente
cuánto del rendimiento proviene de reconocer la pregunta en lugar de entender la explicación.

In [ ]:
fig, axes = plt.subplots(1, len(etapas), figsize=(15, 4.6), sharey=True)

for ax, etapa in zip(np.atleast_1d(axes), etapas):
    p = (
        metricas[metricas["etapa"] == etapa]
        .pivot_table(index="modelo", columns="esquema", values="map@3")
        .reindex(orden_modelos)
    )
    brecha = (p["aleatorio"] - p["agrupado"]).fillna(0)
    colores = [COLOR_AGRUPADO if v > 0.1 else COLOR_ALEATORIO for v in brecha]
    ax.barh(brecha.index, brecha.values, color=colores)
    for y, v in enumerate(brecha.values):
        ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=8.5)
    ax.set_xlabel("Caída de MAP@3")
    ax.set_title(etapa)
    ax.set_xlim(0, max(0.05, brecha.max() * 1.3))

fig.suptitle("Pérdida de rendimiento al enfrentar preguntas nunca vistas",
             fontsize=13, fontweight="bold")
fig.tight_layout()
print(guardar_figura(fig, "fig2_brecha_validacion"))
plt.show()

## 4. Figura 3: cuánto aporta leer la explicación

Se contrasta el mejor modelo que lee el texto contra la línea base que solo conoce la pregunta y
la opción marcada.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)

for ax, esquema in zip(axes, ["aleatorio", "agrupado"]):
    sub = comparacion[comparacion["esquema"] == esquema].sort_values("etapa")
    pos = np.arange(len(sub))
    ancho = 0.36
    ax.bar(pos - ancho / 2, sub["map3_con_texto"], ancho,
           color=COLOR_TEXTO, label="Lee la explicación")
    ax.bar(pos + ancho / 2, sub["map3_solo_contexto"], ancho,
           color=COLOR_CONTEXTO, label="Solo pregunta y respuesta")
    for x, (a, b) in enumerate(zip(sub["map3_con_texto"], sub["map3_solo_contexto"])):
        ax.text(x - ancho / 2, a + 0.012, f"{a:.3f}", ha="center", fontsize=8.5)
        ax.text(x + ancho / 2, b + 0.012, f"{b:.3f}", ha="center", fontsize=8.5)
    ax.set_xticks(pos)
    ax.set_xticklabels(sub["etapa"], fontsize=9)
    ax.set_ylabel("MAP@3")
    ax.set_title(ETIQUETA_ESQUEMA[esquema].replace("\n", " "))
    ax.set_ylim(0, 1.05)

axes[0].legend(loc="upper right", fontsize=9)
fig.suptitle("Aporte real del texto escrito por el estudiante", fontsize=13, fontweight="bold")
fig.tight_layout()
print(guardar_figura(fig, "fig3_aporte_del_texto"))
plt.show()

In [ ]:
comparacion

## 5. Figura 4: rendimiento frente a costo de cómputo

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5))

sub = metricas[(metricas["etapa"] == "3. Compuesto")]
for i, modelo in enumerate(orden_modelos):
    d = sub[sub["modelo"] == modelo]
    for esquema, marcador in [("aleatorio", "o"), ("agrupado", "s")]:
        fila = d[d["esquema"] == esquema]
        if fila.empty:
            continue
        ax.scatter(fila["segundos"], fila["map@3"], s=130, marker=marcador,
                   color=PALETA_CUALITATIVA[i % len(PALETA_CUALITATIVA)],
                   edgecolor="white", linewidth=1.2, zorder=3,
                   label=modelo if esquema == "aleatorio" else None)

ax.set_xscale("log")
ax.set_xlabel("Segundos de validación cruzada (escala logarítmica)")
ax.set_ylabel("MAP@3 del objetivo compuesto")
ax.set_title("Un modelo más costoso no siempre acierta más")
ax.legend(fontsize=8.5, loc="lower right", title="Modelo", title_fontsize=9)
ax.text(0.02, 0.97, "círculo: validación aleatoria\ncuadrado: agrupada por pregunta",
        transform=ax.transAxes, va="top", fontsize=8.5,
        bbox={"facecolor": "white", "edgecolor": "#D6DBE0"})
fig.tight_layout()
print(guardar_figura(fig, "fig4_rendimiento_vs_costo"))
plt.show()

## 6. Figura 5: el efecto del desbalance

Se relaciona la frecuencia de cada error conceptual con el número de preguntas en que aparece.
Esto conecta el desbalance observado en el análisis exploratorio con la dificultad que muestran
los modelos en la etapa 2.

In [ ]:
con_error = train[train["misconception"] != "NA"].copy()
con_error["etiqueta"] = con_error["misconception"].replace({"Wrong_Fraction": "Wrong_fraction"})

perfil = (
    con_error.groupby("etiqueta")
    .agg(frecuencia=("row_id", "size"), preguntas=("QuestionId", "nunique"))
    .reset_index()
)

fig, (izq, der) = plt.subplots(1, 2, figsize=(14, 5))

colores = [COLOR_AGRUPADO if p == 1 else COLOR_TEXTO for p in perfil["preguntas"]]
izq.scatter(perfil["frecuencia"], perfil["preguntas"], s=90, c=colores,
            alpha=0.85, edgecolor="white", linewidth=1)
izq.set_xscale("log")
izq.set_yticks(sorted(perfil["preguntas"].unique()))
izq.set_xlabel("Observaciones de la etiqueta (escala logarítmica)")
izq.set_ylabel("Preguntas distintas en que aparece")
izq.set_title("Casi toda etiqueta vive en una sola pregunta")
for _, f in perfil.nlargest(4, "frecuencia").iterrows():
    izq.annotate(f["etiqueta"], (f["frecuencia"], f["preguntas"]),
                 textcoords="offset points", xytext=(6, 7), fontsize=8)

acumulado = (
    perfil.sort_values("frecuencia", ascending=False)["frecuencia"].cumsum()
    / perfil["frecuencia"].sum() * 100
)
der.plot(range(1, len(acumulado) + 1), acumulado.values, marker="o", ms=4.5,
         color=COLOR_ALEATORIO)
der.axhline(80, color=COLOR_AGRUPADO, ls="--", lw=1.3, label="80 % de los casos")
der.set_xlabel("Etiquetas ordenadas por frecuencia")
der.set_ylabel("Porcentaje acumulado")
der.set_title("Unas pocas etiquetas concentran casi todo")
der.legend(fontsize=9)

fig.suptitle("Por qué la etapa 2 es tan difícil", fontsize=13, fontweight="bold")
fig.tight_layout()
print(guardar_figura(fig, "fig5_desbalance_y_cobertura"))
plt.show()

print(f"\nEtiquetas presentes en una sola pregunta: "
      f"{(perfil['preguntas'] == 1).sum()} de {len(perfil)}")

## 7. Tabla resumen para el informe

In [ ]:
resumen = (
    metricas.pivot_table(
        index=["etapa", "modelo"], columns="esquema", values=["map@3", "f1_macro"]
    )
    .round(4)
)
resumen.columns = [f"{m}_{e}" for m, e in resumen.columns]
resumen = resumen.reset_index()
resumen.to_csv(DATA_TRANSFORMED / "resumen_informe.csv", index=False)
print("Guardado en data/transformed/resumen_informe.csv")
resumen

In [ ]:
print("Figuras generadas en reports/figures/:")
for p in sorted(FIGURES.glob("*.png")):
    print(f"  {p.name:38s} {p.stat().st_size / 1024:6.0f} KB")

## 8. Hallazgos de la comparación

Las celdas anteriores sostienen las conclusiones que se desarrollan en el informe final. Las
figuras quedan guardadas y listas para incorporarse a él.